# Alert Correlator — W2D1 Assignment

Pipeline: **Dedup → Time-Window → Topology → Combined Correlation**

- `gap_sec = 120` (2 phút) — vì toàn bộ incident window chỉ ~6.5 phút, gap lớn nhất giữa 2 alert liên tiếp là 49s.
- `max_hop = 2` — vì cascade thường lan qua 1-2 hop trên service graph.

In [1]:
import json
import os
import networkx as nx
from datetime import datetime, timezone
from collections import defaultdict

# ── Configuration ──
GAP_SEC  = 120   # time-window gap threshold (seconds)
MAX_HOP  = 2     # topology grouping max hop distance

DATASET_DIR = 'dataset'
RESULTS_DIR = 'results'

SEVERITY_ORDER = {'info': 0, 'warn': 1, 'crit': 2}

print('Configuration loaded.')
print(f'  GAP_SEC  = {GAP_SEC}')
print(f'  MAX_HOP  = {MAX_HOP}')

Configuration loaded.
  GAP_SEC  = 120
  MAX_HOP  = 2


In [2]:
# ═══════════════════════════════════════════════════════
# Cell 2: Load Data — alerts + service dependency graph
# ═══════════════════════════════════════════════════════

def load_alerts(path: str) -> list:
    """Load alerts from JSONL, parse timestamps, sort by time."""
    alerts = []
    with open(path, 'r', encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if line:
                alerts.append(json.loads(line))
    for a in alerts:
        a['_ts'] = datetime.fromisoformat(a['ts'].replace('Z', '+00:00'))
    alerts.sort(key=lambda a: a['_ts'])
    return alerts


def load_graph(path: str) -> nx.DiGraph:
    """Build directed graph from services.json."""
    with open(path, 'r', encoding='utf-8') as f:
        data = json.load(f)
    G = nx.DiGraph()
    for svc in data.get('services', []):
        G.add_node(svc['name'], **{k: v for k, v in svc.items() if k != 'name'})
    for store in data.get('stores', []):
        G.add_node(store['name'], **{k: v for k, v in store.items() if k != 'name'})
    for edge in data.get('edges', []):
        G.add_edge(edge['from'], edge['to'], type=edge.get('type', 'unknown'))
    return G


# Load
alerts = load_alerts(os.path.join(DATASET_DIR, 'alerts_sample.jsonl'))
graph  = load_graph(os.path.join(DATASET_DIR, 'services.json'))

print(f'Loaded {len(alerts)} alerts')
print(f'Graph: {graph.number_of_nodes()} nodes, {graph.number_of_edges()} edges')
print()
print('First 5 alerts:')
for a in alerts[:5]:
    print(f"  {a['id']} | {a['ts']} | {a['service']:20s} | {a['metric']:35s} | {a['severity']}")

Loaded 20 alerts
Graph: 14 nodes, 17 edges

First 5 alerts:
  a-0001 | 2026-06-12T09:42:01Z | payment-svc          | db_connection_pool_used_ratio       | warn
  a-0002 | 2026-06-12T09:42:18Z | payment-svc          | db_connection_pool_used_ratio       | crit
  a-0003 | 2026-06-12T09:42:22Z | payment-svc          | latency_p99_ms                      | crit
  a-0004 | 2026-06-12T09:42:30Z | payment-svc          | error_rate                          | warn
  a-0005 | 2026-06-12T09:42:45Z | checkout-svc         | latency_p99_ms                      | warn


In [3]:
# ═══════════════════════════════════════════════════════
# Cell 3: Layer 1 — Fingerprint Dedup
# ═══════════════════════════════════════════════════════

def fingerprint(alert: dict) -> str:
    """Stable fingerprint: service|metric|severity.
    
    Does NOT include timestamp or value because:
    - timestamp changes every fire → no dedup at all
    - value fluctuates around threshold → miss near-duplicate alerts
    """
    return f"{alert['service']}|{alert['metric']}|{alert['severity']}"


def dedup(alerts: list) -> list:
    """Keep the first occurrence of each fingerprint."""
    seen = set()
    result = []
    for a in alerts:
        fp = fingerprint(a)
        a['fingerprint'] = fp
        if fp not in seen:
            seen.add(fp)
            result.append(a)
    return result


# Tag all alerts with fingerprint (even if we don't remove them)
for a in alerts:
    a['fingerprint'] = fingerprint(a)

deduped = dedup(alerts)

print(f'Before dedup: {len(alerts)} alerts')
print(f'After  dedup: {len(deduped)} unique fingerprints')
print(f'Removed: {len(alerts) - len(deduped)} duplicate(s)')
print()

# Show duplicates
fp_counts = defaultdict(int)
for a in alerts:
    fp_counts[a['fingerprint']] += 1

print('Duplicate fingerprints:')
for fp, count in fp_counts.items():
    if count > 1:
        ids = [a['id'] for a in alerts if a['fingerprint'] == fp]
        print(f'  {fp} (x{count}) → IDs: {ids}')

Before dedup: 20 alerts
After  dedup: 17 unique fingerprints
Removed: 3 duplicate(s)

Duplicate fingerprints:
  payment-svc|db_connection_pool_used_ratio|crit (x2) → IDs: ['a-0002', 'a-0011']
  payment-svc|latency_p99_ms|crit (x3) → IDs: ['a-0003', 'a-0008', 'a-0015']


In [4]:
# ═══════════════════════════════════════════════════════
# Cell 4: Layer 2 — Time-Window Session Grouping
# ═══════════════════════════════════════════════════════

def session_groups(alerts: list, gap_sec: int = 120) -> list:
    """Group alerts into sessions: consecutive alerts ≤ gap_sec apart."""
    if not alerts:
        return []
    alerts_sorted = sorted(alerts, key=lambda a: a['_ts'])
    sessions = [[alerts_sorted[0]]]
    for a in alerts_sorted[1:]:
        if (a['_ts'] - sessions[-1][-1]['_ts']).total_seconds() <= gap_sec:
            sessions[-1].append(a)
        else:
            sessions.append([a])
    return sessions


sessions = session_groups(alerts, gap_sec=GAP_SEC)
print(f'Sessions found: {len(sessions)} (gap_sec={GAP_SEC})')
print()
for i, sess in enumerate(sessions):
    svcs = sorted({a['service'] for a in sess})
    print(f'Session {i}: {len(sess)} alerts')
    print(f'  Time: {sess[0]["ts"]} → {sess[-1]["ts"]}')
    print(f'  Services: {svcs}')

# Also show what happens with smaller gap
print()
print('--- Comparison: gap_sec=30 ---')
sessions_30 = session_groups(alerts, gap_sec=30)
print(f'Sessions with gap_sec=30: {len(sessions_30)}')
for i, sess in enumerate(sessions_30):
    print(f'  Session {i}: {len(sess)} alerts | {sess[0]["ts"]} → {sess[-1]["ts"]}')

Sessions found: 1 (gap_sec=120)

Session 0: 20 alerts
  Time: 2026-06-12T09:42:01Z → 2026-06-12T09:48:30Z
  Services: ['cart-svc', 'checkout-svc', 'edge-lb', 'notification-svc', 'payment-svc', 'recommender-svc', 'search-svc']

--- Comparison: gap_sec=30 ---
Sessions with gap_sec=30: 5
  Session 0: 12 alerts | 2026-06-12T09:42:01Z → 2026-06-12T09:44:30Z
  Session 1: 1 alerts | 2026-06-12T09:45:10Z → 2026-06-12T09:45:10Z
  Session 2: 2 alerts | 2026-06-12T09:45:42Z → 2026-06-12T09:46:01Z
  Session 3: 2 alerts | 2026-06-12T09:46:50Z → 2026-06-12T09:47:12Z
  Session 4: 3 alerts | 2026-06-12T09:48:01Z → 2026-06-12T09:48:30Z


In [5]:
# ═══════════════════════════════════════════════════════
# Cell 5: Layer 3 — Topology Grouping + Combined Correlation
# ═══════════════════════════════════════════════════════

def topology_group(alerts: list, graph, max_hop: int = 2) -> list:
    """Group alerts whose services are ≤ max_hop apart on undirected graph."""
    undirected = graph.to_undirected()
    by_service = defaultdict(list)
    for a in alerts:
        by_service[a['service']].append(a)

    services = list(by_service.keys())
    parent = {s: s for s in services}

    def find(x):
        while parent[x] != x:
            parent[x] = parent[parent[x]]
            x = parent[x]
        return x

    for i, s1 in enumerate(services):
        for s2 in services[i + 1:]:
            try:
                if nx.shortest_path_length(undirected, s1, s2) <= max_hop:
                    parent[find(s1)] = find(s2)
            except nx.NetworkXNoPath:
                pass

    groups = defaultdict(list)
    for s in services:
        groups[find(s)].extend(by_service[s])
    return list(groups.values())


def correlate(alerts: list, graph, gap_sec: int = 120, max_hop: int = 2) -> list:
    """Full pipeline: time-window + topology grouping."""
    sessions = session_groups(alerts, gap_sec=gap_sec)
    clusters = []
    for s_idx, session_alerts in enumerate(sessions):
        for g_idx, group in enumerate(topology_group(session_alerts, graph, max_hop)):
            clusters.append({
                'cluster_id': f'c-{s_idx:03d}-{g_idx:03d}',
                'alert_count': len(group),
                'services': sorted({a['service'] for a in group}),
                'time_range': [
                    min(a['ts'] for a in group),
                    max(a['ts'] for a in group),
                ],
                'max_severity': max(
                    (a['severity'] for a in group),
                    key=lambda s: SEVERITY_ORDER.get(s, -1)
                ),
                'fingerprints': sorted({a['fingerprint'] for a in group}),
                'alert_ids': [a['id'] for a in group],
            })
    return clusters


# Run the combined correlator
clusters = correlate(alerts, graph, gap_sec=GAP_SEC, max_hop=MAX_HOP)

print(f'Combined Correlation (gap_sec={GAP_SEC}, max_hop={MAX_HOP})')
print(f'Total clusters: {len(clusters)}')
print()
for c in clusters:
    print(f"Cluster {c['cluster_id']}:")
    print(f"  Alerts: {c['alert_count']}")
    print(f"  Services: {c['services']}")
    print(f"  Time: {c['time_range'][0]} → {c['time_range'][1]}")
    print(f"  Max severity: {c['max_severity']}")
    print(f"  Alert IDs: {c['alert_ids']}")
    print(f"  Fingerprints: {c['fingerprints']}")
    print()

Combined Correlation (gap_sec=120, max_hop=2)
Total clusters: 1

Cluster c-000-000:
  Alerts: 20
  Services: ['cart-svc', 'checkout-svc', 'edge-lb', 'notification-svc', 'payment-svc', 'recommender-svc', 'search-svc']
  Time: 2026-06-12T09:42:01Z → 2026-06-12T09:48:30Z
  Max severity: crit
  Alert IDs: ['a-0001', 'a-0002', 'a-0003', 'a-0004', 'a-0008', 'a-0011', 'a-0015', 'a-0018', 'a-0005', 'a-0006', 'a-0012', 'a-0017', 'a-0007', 'a-0014', 'a-0020', 'a-0009', 'a-0010', 'a-0019', 'a-0013', 'a-0016']
  Fingerprints: ['cart-svc|latency_p99_ms|warn', 'checkout-svc|downstream_payment_error_rate|crit', 'checkout-svc|latency_p99_ms|crit', 'checkout-svc|latency_p99_ms|warn', 'checkout-svc|request_drop_rate|crit', 'edge-lb|p99_latency_ms|crit', 'edge-lb|upstream_5xx_rate|crit', 'edge-lb|upstream_5xx_rate|warn', 'notification-svc|queue_depth|crit', 'notification-svc|queue_lag_ms|warn', 'payment-svc|db_connection_pool_used_ratio|crit', 'payment-svc|db_connection_pool_used_ratio|warn', 'payment-sv

In [6]:
# ═══════════════════════════════════════════════════════
# Cell 6: Write results/cluster_summary.json
# ═══════════════════════════════════════════════════════

total_alerts   = len(alerts)
total_clusters = len(clusters)
reduction      = round(1 - total_clusters / total_alerts, 4)

summary = {
    'input_alerts': total_alerts,
    'output_clusters': total_clusters,
    'reduction_ratio': reduction,
    'clusters': clusters,
}

os.makedirs(RESULTS_DIR, exist_ok=True)
out_path = os.path.join(RESULTS_DIR, 'cluster_summary.json')

# Remove internal _ts before writing
clean_clusters = []
for c in clusters:
    clean_clusters.append({k: v for k, v in c.items()})

summary['clusters'] = clean_clusters

with open(out_path, 'w', encoding='utf-8') as f:
    json.dump(summary, f, indent=2, ensure_ascii=False)

print('=== cluster_summary.json ===')
print(f'  input_alerts    = {total_alerts}')
print(f'  output_clusters = {total_clusters}')
print(f'  reduction_ratio = {reduction}')
print(f'  Written to: {out_path}')
print()
print('--- Full JSON output ---')
print(json.dumps(summary, indent=2, ensure_ascii=False))

=== cluster_summary.json ===
  input_alerts    = 20
  output_clusters = 1
  reduction_ratio = 0.95
  Written to: results\cluster_summary.json

--- Full JSON output ---
{
  "input_alerts": 20,
  "output_clusters": 1,
  "reduction_ratio": 0.95,
  "clusters": [
    {
      "cluster_id": "c-000-000",
      "alert_count": 20,
      "services": [
        "cart-svc",
        "checkout-svc",
        "edge-lb",
        "notification-svc",
        "payment-svc",
        "recommender-svc",
        "search-svc"
      ],
      "time_range": [
        "2026-06-12T09:42:01Z",
        "2026-06-12T09:48:30Z"
      ],
      "max_severity": "crit",
      "fingerprints": [
        "cart-svc|latency_p99_ms|warn",
        "checkout-svc|downstream_payment_error_rate|crit",
        "checkout-svc|latency_p99_ms|crit",
        "checkout-svc|latency_p99_ms|warn",
        "checkout-svc|request_drop_rate|crit",
        "edge-lb|p99_latency_ms|crit",
        "edge-lb|upstream_5xx_rate|crit",
        "edge-lb|upstre